In [2]:
from importlib.metadata import version

print("torch version: ", version("torch"))
print("transformers version: ", version("transformers"))
print("datasets version: ", version("datasets"))

torch version:  2.7.0
transformers version:  4.51.3
datasets version:  2.18.0


In [3]:
import torch
import transformers
import datasets
import json
import math

In [4]:
# 加载分词器
tokenizer = transformers.AutoTokenizer.from_pretrained(
    "tokenizers/merged_tokenizer/"
)

In [5]:
tokenizer.vocab_size

58703

In [6]:
# 重新定义模型的结构参数
context_length = 512
config = transformers.AutoConfig.from_pretrained(
    "tokenizers/Qwen/Qwen3-0.6B/",
    bos_token_id=1,
    eos_token_id=2,
    hidden_size=512,
    intermediate_size=1536,
    max_position_embeddings=8192,
    num_attention_heads=8,
    num_key_value_heads=4,
    num_hidden_layers=16,
    rope_theta=10000,
    vocab_size=58703,
    n_ctx=context_length
)

In [7]:
print(config)

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 512,
  "initializer_range": 0.02,
  "intermediate_size": 1536,
  "max_position_embeddings": 8192,
  "max_window_layers": 28,
  "model_type": "qwen3",
  "num_attention_heads": 8,
  "num_hidden_layers": 16,
  "num_key_value_heads": 4,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 10000,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.51.3",
  "use_cache": true,
  "use_sliding_window": false,
  "vocab_size": 58703
}



In [9]:
# embedding层的参数
# 150000 * 1024 * 2 / 1e6
58703 * 512 * 2 / 1e6

60.111872

In [10]:
# 定义模型
model = transformers.Qwen3ForCausalLM(config)
print("Model Summary:")
print(model)

Model Summary:
Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(58703, 512)
    (layers): ModuleList(
      (0-15): 16 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=512, out_features=1024, bias=False)
          (k_proj): Linear(in_features=512, out_features=512, bias=False)
          (v_proj): Linear(in_features=512, out_features=512, bias=False)
          (o_proj): Linear(in_features=1024, out_features=512, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=512, out_features=1536, bias=False)
          (up_proj): Linear(in_features=512, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=512, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((512,), eps=1e-06)
        (post_attention_layernorm): 

In [11]:
# 打印所有参数的shape和名称
print("Model Parameters and Shape:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")

Model Parameters and Shape:
model.embed_tokens.weight: torch.Size([58703, 512])
model.layers.0.self_attn.q_proj.weight: torch.Size([1024, 512])
model.layers.0.self_attn.k_proj.weight: torch.Size([512, 512])
model.layers.0.self_attn.v_proj.weight: torch.Size([512, 512])
model.layers.0.self_attn.o_proj.weight: torch.Size([512, 1024])
model.layers.0.self_attn.q_norm.weight: torch.Size([128])
model.layers.0.self_attn.k_norm.weight: torch.Size([128])
model.layers.0.mlp.gate_proj.weight: torch.Size([1536, 512])
model.layers.0.mlp.up_proj.weight: torch.Size([1536, 512])
model.layers.0.mlp.down_proj.weight: torch.Size([512, 1536])
model.layers.0.input_layernorm.weight: torch.Size([512])
model.layers.0.post_attention_layernorm.weight: torch.Size([512])
model.layers.1.self_attn.q_proj.weight: torch.Size([1024, 512])
model.layers.1.self_attn.k_proj.weight: torch.Size([512, 512])
model.layers.1.self_attn.v_proj.weight: torch.Size([512, 512])
model.layers.1.self_attn.o_proj.weight: torch.Size([512,

In [113]:
# 加载预训练数据
raw_datasets = datasets.load_dataset(
    "json", data_files="data/processed_dataset_512_clean_230w.jsonl"
)
print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2299107
    })
})


In [114]:
raw_datasets = raw_datasets["train"].train_test_split(test_size=0.005, seed=42)
print("dataset info:")
print(raw_datasets)

dataset info:
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2287611
    })
    test: Dataset({
        features: ['text'],
        num_rows: 11496
    })
})


In [115]:
# 保存测试数据
raw_datasets["test"].to_json(path_or_buf='./data/pretrain_test_1w.json', force_ascii=False)

Creating json from Arrow format:   0%|          | 0/12 [00:00<?, ?ba/s]

10539929

In [116]:
# 预训练数据处理
def tokenize(element):
    outputs = tokenizer(
        element['text'],
        add_special_tokens=False
    )
    input_ids_list = outputs['input_ids']
    new_input_ids_list, new_attn_mask_list = [], []
    for input_ids in input_ids_list:
        input_ids_eos = input_ids[:context_length-1] + [tokenizer.eos_token_id]
        new_input_ids_list.append(input_ids_eos)
        new_attn_mask_list.append([1] * len(input_ids_eos))
    return {
        "input_ids": new_input_ids_list,
        "attention_mask": new_attn_mask_list
    }

In [117]:
raw_datasets["train"][0]

{'text': '用英文怎么说:\n利比亚的绿山完全没有受到污染,有丰沛的土壤、峡谷,及希腊的遗迹。The Green Mountain in Libya is a virtually unspoiled region of fertile land, gorges and Greek ruins.'}

In [118]:
raw_datasets["train"][:2]

{'text': ['用英文怎么说:\n利比亚的绿山完全没有受到污染,有丰沛的土壤、峡谷,及希腊的遗迹。The Green Mountain in Libya is a virtually unspoiled region of fertile land, gorges and Greek ruins.',
  '根据提供的主题/问题,从所提供的文本集合中抽取相关信息。\n主题:奶制品贸易争端。文本:全球乳制品巨头、新西兰的丹尼斯·高特来到北京的开幕式上,到场的还有知名企业家等。在一个有动画和媒体展示的舞台上,高特以易于理解和独特的方式分享了关于中国和新西兰之间的乳制品贸易争端的信息。根据提供的文本,可以知道以下信息:\n- 丹尼斯·高特是全球乳制品巨头,来到北京的开幕式上。\n- 在开幕式上,有知名企业家等人到场。\n- 丹尼斯·高特在一个有动画和媒体展示的舞台上,以易于理解和独特的方式分享了关于中国和新西兰之间的乳制品贸易争端的信息。\n- 根据文本内容,无法确定乳制品贸易争端的具体细节和背景信息。']}

In [119]:
print(tokenizer(raw_datasets["train"][:2]["text"]))
print(tokenizer(raw_datasets["train"][:2]["text"]).keys())

{'input_ids': [[339, 2374, 1811, 291, 50551, 118, 123, 36, 209, 642, 4822, 270, 1660, 739, 2717, 925, 1851, 2189, 22, 320, 1878, 407, 260, 270, 5912, 300, 6006, 2233, 22, 560, 4483, 270, 2074, 2956, 272, 2598, 12229, 18317, 1696, 22120, 2205, 1436, 18504, 5595, 8804, 89, 12655, 12569, 1721, 39735, 10799, 22, 2628, 11211, 908, 1848, 17005, 28681, 24], [7233, 2940, 25, 556, 22, 598, 531, 716, 2755, 6650, 314, 4017, 9350, 272, 209, 1145, 36, 1914, 6466, 4737, 1376, 1762, 272, 634, 36, 1259, 2439, 6466, 1948, 878, 300, 490, 7675, 270, 2536, 6543, 646, 176, 54332, 512, 3999, 2286, 270, 522, 2831, 565, 385, 22, 389, 664, 270, 1910, 3575, 7331, 414, 272, 3701, 320, 4092, 292, 1684, 2978, 270, 6674, 385, 22, 176, 54332, 512, 304, 3709, 3687, 2550, 2697, 2948, 310, 1404, 885, 292, 490, 7675, 1644, 2439, 6466, 4737, 1376, 1762, 2551, 272, 7233, 2755, 22, 381, 1670, 626, 904, 36, 209, 23, 231, 2536, 6543, 646, 176, 54332, 512, 9295, 2439, 6466, 1948, 878, 22, 3999, 2286, 270, 522, 2831, 565, 385,

In [120]:
# tokenizer原始文本数据
tokenized_datasets = raw_datasets.map(
    tokenize, batched=True, num_proc=16, remove_columns=raw_datasets["train"].column_names
)

Map (num_proc=16):   0%|          | 0/2287611 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/11496 [00:00<?, ? examples/s]

In [121]:
print("tokenized dataset info:")
print(tokenized_datasets)

tokenized dataset info:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2287611
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 11496
    })
})


In [122]:
input_examples = [
    {"input_ids": [5714, 876, 272, 2], 'attention_mask': [1, 1, 1, 1]},
    {'input_ids': [22, 1491, 1664, 4532, 292, 4571, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}
]
# input: a, b, c, d
# label: b, c, d, <eos>

# 测试data_collator
data_collator = transformers.DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(data_collator(input_examples))
# 底层自动移一位

{'input_ids': tensor([[5714,  876,  272,    2,    0,    0,    0],
        [  22, 1491, 1664, 4532,  292, 4571,    2]]), 'attention_mask': tensor([[1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[5714,  876,  272,    2, -100, -100, -100],
        [  22, 1491, 1664, 4532,  292, 4571,    2]])}


In [125]:
# 训练参数

args = transformers.TrainingArguments(
    output_dir='saved/',
    per_device_train_batch_size=16, # 每个GPU的训练batch数
    per_device_eval_batch_size=16, # 每个GPU的验证batch数
    gradient_accumulation_steps=8, # 梯度的累积步数
    eval_strategy='steps',
    eval_steps=10,
    logging_steps=5,
    num_train_epochs=2, # 训练的epochs数
    weight_decay=0.1,   # weight decay的比率
    optim='adamw_torch', # 优化器选择AdamW
    warmup_ratio=0.1,   # warmup的比率
    lr_scheduler_type='cosine', # 学习率的衰减策略, [0, T/4]
    learning_rate=3e-4, # 学习率，[1e-4, 5e-5]
    save_steps=500,
    save_total_limit=2, # 最大保存5个ckpt
    bf16=True, # 开启bf16训练，对于Amper架构以下的显卡建议替换为fp16
)
print("Train Args:")
print(args)

Train Args:
TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=10,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_obj

In [ ]:
# 开始训练
trainer = transformers.Trainer(
    model=model,
    processing_class=tokenizer,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets['test'],
    data_collator=data_collator
)
trainer.train()

In [127]:
# 评估的代码
eval_results = trainer.evaluate()
print(eval_results)

{'eval_loss': 9.101799964904785}


In [128]:
print(f"Perplexity: {math.exp(eval_results['eval_loss'])}")

Perplexity: 8971.42643177728


In [129]:
# 根据提示词生成答案
pipe = transformers.pipeline("text-generation", model=model, tokenizer=tokenizer)
pipe("人工智能", num_return_sequences=1)

Device set to use cuda:0


[{'generated_text': '人工智能,,,,,,,,,,,,,,,,,,,,'}]